**Prediction of Burnout in Remote Work Settings**

---

**Augmentation Notebook Version**


In [17]:
#############################
#       IMPORT LIBRARIES    #
#############################

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV

from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/ML Group Project CSCI 635')

from analysis.data_splits import train_test


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
#############################
#     DEFINE RANDOM SEED    #
#############################

seed = 11
rand_state = 11

np.random.seed(seed)
print("Seed set to:", seed)


Seed set to: 11


In [19]:

#############################
#     LOAD TRAINING DATA    #
#############################

def get_unscaled_train_data():
    # Fetch raw training dataset from data_splits.py
    X_train, X_test, y_train, y_test = train_test()
    return X_train.reset_index(drop=True), y_train.squeeze().reset_index(drop=True)

def get_scaled_train_data():
    X_train_raw, y_train = get_unscaled_train_data()

    sc = StandardScaler()
    X_train_scaled = sc.fit_transform(X_train_raw)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_raw.columns)

    return X_train_scaled, y_train

X_raw, y_raw = get_unscaled_train_data()
X, y = get_scaled_train_data()

print("Scaled X shape:", X.shape)
print("y shape:", y.shape)
print("Raw X shape:", X_raw.shape)
X.head()


Scaled X shape: (1600, 11)
y shape: (1600,)
Raw X shape: (1600, 11)


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,0.059892,-0.190374,-0.244904,1.097410,-0.431351,0.316649,-0.744093,-0.079895,-1.139238,-0.273703,-0.628445
1,-0.438256,-0.622754,-0.590752,1.097410,-0.431351,-1.321600,0.235483,0.891810,-1.139238,-0.359298,-0.628445
2,-1.239874,-1.070043,0.100944,-0.027778,-0.431351,-0.943542,0.134496,0.894133,-1.139238,-1.246781,-0.628445
3,0.145780,0.256915,-0.936600,-0.590372,-0.431351,0.022605,0.487951,0.386988,0.079198,-0.949452,1.591229
4,1.245142,0.962638,1.138489,0.534816,-0.431351,1.996905,-1.521695,-0.912231,1.297635,1.546313,-0.628445


In [20]:

#############################
#   SYNTHETIC DATA METHODS  #
#############################

def _apply_constraints(df, constraints):
    if not constraints:
        return df

    df = df.copy()

    for col, spec in constraints.items():
        if col not in df.columns:
            continue

        vals = df[col].astype(float).to_numpy()
        t = spec.get("type")

        if t == "categorical":
            allowed = np.array(spec.get("values", []))
            if allowed.size == 0:
                continue
            idx = np.abs(vals[:, None] - allowed[None, :]).argmin(axis=1)
            df[col] = allowed[idx]

        elif t == "int":
            mn = spec.get("min", None)
            mx = spec.get("max", None)
            ints = np.rint(vals).astype(int)

            if mn is not None or mx is not None:
                if mn is None:
                    mn = ints.min()
                if mx is None:
                    mx = ints.max()
                ints = np.clip(ints, mn, mx)

            df[col] = ints

        elif t == "float":
            mn = spec.get("min", None)
            mx = spec.get("max", None)
            fvals = vals

            if mn is not None or mx is not None:
                if mn is None:
                    mn = fvals.min()
                if mx is None:
                    mx = fvals.max()
                fvals = np.clip(fvals, mn, mx)

            df[col] = fvals

        else:
            mn = spec.get("min", None)
            mx = spec.get("max", None)
            if mn is not None or mx is not None:
                df[col] = np.clip(
                    vals,
                    mn if mn is not None else vals.min(),
                    mx if mx is not None else vals.max()
                )

    return df

def _data_gen(X, y, n_samples=2000, constraints=None):
    y = y.squeeze()
    y_arr = y.values.reshape(-1)
    classes, counts = np.unique(y_arr, return_counts=True)

    kdes = {}
    bandwidth_params = {'bandwidth': np.arange(0.01, 1, 0.05)}

    for cls in classes:
        X_cls = X[y_arr == cls]
        grid_search = GridSearchCV(KernelDensity(), bandwidth_params, cv=5)
        grid_search.fit(X_cls.values)
        kde = grid_search.best_estimator_
        kdes[cls] = kde

    freqs = counts / counts.sum()
    samples_per_class = np.floor(freqs * n_samples).astype(int)

    remainder = n_samples - samples_per_class.sum()
    if remainder > 0:
        idxs = np.argsort(-counts)
        for i in range(remainder):
            samples_per_class[idxs[i % len(classes)]] += 1

    new_X = []
    new_y = []

    for cls, n in zip(classes, samples_per_class):
        new_data = kdes[cls].sample(n, random_state=rand_state)
        new_X.append(new_data)
        new_y.append(np.full(n, cls))

    X_synthset = np.vstack(new_X)
    y_synthset = np.concatenate(new_y)

    df_X_synth = pd.DataFrame(X_synthset, columns=X.columns)
    df_y_synth = pd.Series(y_synthset)

    df_X_synth = _apply_constraints(df_X_synth, constraints)

    return df_X_synth, df_y_synth

def generate_synthetic_data(X, y, n_samples=2000):
    return _data_gen(X, y, n_samples=n_samples, constraints=None)

def generate_unscaled_synthetic_data(X, y, n_samples=2000):
    constraints = {
        "day_type": {"type": "categorical", "values": [0, 1]},
        "work_hours": {"type": "float", "min": 0.5, "max": 18},
        "screen_time_hours": {"type": "float", "min": 0, "max": 18},
        "meetings_count": {"type": "int", "min": 0, "max": 20},
        "breaks_taken": {"type": "int", "min": 0, "max": 15},
        "after_hours_work": {"type": "categorical", "values": [0, 1]},
        "app_switches": {"type": "int", "min": 5, "max": 200},
        "sleep_hours": {"type": "float", "min": 3, "max": 10},
        "task_completion": {"type": "float", "min": 0, "max": 100},
        "isolation_index": {"type": "int", "min": 3, "max": 9},
        "fatigue_score": {"type": "float", "min": 0, "max": 10},
    }
    return _data_gen(X, y, n_samples=n_samples, constraints=constraints)


In [21]:

#############################
#   GENERATE SYNTHETIC DATA #
#############################

n_samples = 2000

# Existing scaled synthetic generation
df_X_synth, df_y_synth = generate_synthetic_data(X, y, n_samples)

# New unscaled synthetic generation
df_X_synth_unscaled, df_y_synth_unscaled = generate_unscaled_synthetic_data(X_raw, y_raw, n_samples)

print("Scaled synthetic X shape:", df_X_synth.shape)
print("Scaled synthetic y shape:", df_y_synth.shape)
print("Unscaled synthetic X shape:", df_X_synth_unscaled.shape)
print("Unscaled synthetic y shape:", df_y_synth_unscaled.shape)

df_X_synth.head()


Scaled synthetic X shape: (2000, 11)
Scaled synthetic y shape: (2000,)
Unscaled synthetic X shape: (2000, 11)
Unscaled synthetic y shape: (2000,)


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,-0.698164,-0.186150,0.763223,-0.973165,-0.293927,-1.828633,0.153969,-2.634567,-0.720773,-1.378130,-1.490043
1,-1.248939,-0.364158,-0.148354,-0.369748,-0.355925,-1.714569,0.825530,-1.514109,-1.438961,-0.683498,-0.391829
2,0.269760,-0.089368,-0.179965,0.065994,-0.863547,-0.968691,0.481465,0.367090,-0.643284,-0.457909,-0.524930
3,-0.333935,-0.601934,-1.020885,1.582837,-0.329034,-1.153279,0.712534,-0.421939,-0.179597,-0.644150,0.244323
4,-1.090191,-0.736081,0.029294,-0.335494,-0.597638,-0.491250,1.051610,1.221635,-0.926146,-0.446573,-0.430095


In [22]:

#############################
#   CHECK CLASS DISTRIBUTION #
#############################

print("Original class distribution:")
print(y.value_counts(normalize=True).sort_index())

print("\nScaled synthetic class distribution:")
print(df_y_synth.value_counts(normalize=True).sort_index())

print("\nUnscaled synthetic class distribution:")
print(df_y_synth_unscaled.value_counts(normalize=True).sort_index())


Original class distribution:
burnout_risk
0    0.509375
1    0.421250
2    0.069375
Name: proportion, dtype: float64

Scaled synthetic class distribution:
0    0.5095
1    0.4215
2    0.0690
Name: proportion, dtype: float64

Unscaled synthetic class distribution:
0    0.5095
1    0.4215
2    0.0690
Name: proportion, dtype: float64


In [23]:

#############################
#   AUGMENT TRAINING DATA   #
#############################

def augment_training_data(X, y, n_samples=2000):
    """
    Generate synthetic data in the current feature space and append it.
    Best for already-scaled training data.
    """
    df_X_synth, df_y_synth = generate_synthetic_data(X, y, n_samples)

    X_aug = pd.concat([X, df_X_synth], ignore_index=True)
    y_aug = pd.concat([y.squeeze(), df_y_synth], ignore_index=True)

    return X_aug, y_aug

def augment_training_data_unscaled(X, y, n_samples=2000):
    """
    Generate synthetic data on raw/unscaled features using constraints,
    then append it to the original raw training data.
    """
    df_X_synth_unscaled, df_y_synth_unscaled = generate_unscaled_synthetic_data(X, y, n_samples)

    X_aug_unscaled = pd.concat([X, df_X_synth_unscaled], ignore_index=True)
    y_aug_unscaled = pd.concat([y.squeeze(), df_y_synth_unscaled], ignore_index=True)

    return X_aug_unscaled, y_aug_unscaled

# Existing scaled augmentation
X_aug, y_aug = augment_training_data(X, y, n_samples=2000)

# New unscaled augmentation
X_aug_unscaled, y_aug_unscaled = augment_training_data_unscaled(X_raw, y_raw, n_samples=2000)

print("Scaled augmented X shape:", X_aug.shape)
print("Scaled augmented y shape:", y_aug.shape)
print("Unscaled augmented X shape:", X_aug_unscaled.shape)
print("Unscaled augmented y shape:", y_aug_unscaled.shape)


Scaled augmented X shape: (3600, 11)
Scaled augmented y shape: (3600,)
Unscaled augmented X shape: (3600, 11)
Unscaled augmented y shape: (3600,)


In [24]:

#############################
#  PREVIEW OUTPUTS SCALED   #
#############################

print("Scaled Synthetic Features:")
display(df_X_synth.head())

print("\nScaled Synthetic Labels:")
display(df_y_synth.head())

print("\nScaled Augmented Features:")
display(X_aug.head())

print("\nScaled Augmented Labels:")
display(y_aug.head())

Scaled Synthetic Features:


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,-0.698164,-0.186150,0.763223,-0.973165,-0.293927,-1.828633,0.153969,-2.634567,-0.720773,-1.378130,-1.490043
1,-1.248939,-0.364158,-0.148354,-0.369748,-0.355925,-1.714569,0.825530,-1.514109,-1.438961,-0.683498,-0.391829
2,0.269760,-0.089368,-0.179965,0.065994,-0.863547,-0.968691,0.481465,0.367090,-0.643284,-0.457909,-0.524930
3,-0.333935,-0.601934,-1.020885,1.582837,-0.329034,-1.153279,0.712534,-0.421939,-0.179597,-0.644150,0.244323
4,-1.090191,-0.736081,0.029294,-0.335494,-0.597638,-0.491250,1.051610,1.221635,-0.926146,-0.446573,-0.430095



Scaled Synthetic Labels:


,0
0,0
1,0
2,0
3,0
4,0



Scaled Augmented Features:


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,0.059892,-0.190374,-0.244904,1.097410,-0.431351,0.316649,-0.744093,-0.079895,-1.139238,-0.273703,-0.628445
1,-0.438256,-0.622754,-0.590752,1.097410,-0.431351,-1.321600,0.235483,0.891810,-1.139238,-0.359298,-0.628445
2,-1.239874,-1.070043,0.100944,-0.027778,-0.431351,-0.943542,0.134496,0.894133,-1.139238,-1.246781,-0.628445
3,0.145780,0.256915,-0.936600,-0.590372,-0.431351,0.022605,0.487951,0.386988,0.079198,-0.949452,1.591229
4,1.245142,0.962638,1.138489,0.534816,-0.431351,1.996905,-1.521695,-0.912231,1.297635,1.546313,-0.628445



Scaled Augmented Labels:


,0
0,0
1,0
2,0
3,0
4,1


In [25]:
#############################
# PREVIEW OUTPUTS UNSCALED  #
#############################

print("\nUnscaled Synthetic Features:")
display(df_X_synth_unscaled.head())

print("\nUnscaled Synthetic Labels:")
display(df_y_synth_unscaled.head())

print("\nUnscaled Augmented Features:")
display(X_aug_unscaled.head())

print("\nUnscaled Augmented Labels:")
display(y_aug_unscaled.head())


Unscaled Synthetic Features:


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,7.433685,7.321264,6,3,0,24,6.353942,48.661473,4,3.220286,-2.297593
1,6.260012,7.110071,3,4,0,16,7.619941,59.243433,3,5.110892,0.630975
2,9.830404,7.465496,3,6,0,30,7.379633,82.852346,4,5.601730,0.276042
3,8.199476,6.329617,1,8,0,30,7.961957,72.852656,4,5.103075,2.327383
4,6.936249,6.193096,4,5,0,44,8.409050,92.685527,3,5.402443,0.528933



Unscaled Synthetic Labels:


,0
0,0
1,0
2,0
3,0
4,0



Unscaled Augmented Features:


,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,app_switches,sleep_hours,task_completion,isolation_index,fatigue_score,day_type_Weekend
0,9.06,7.31,3,7,0,64,6.16,79.01,3,5.95,0.0
1,8.19,6.44,2,7,0,25,7.13,91.56,3,5.76,0.0
2,6.79,5.54,4,5,0,34,7.03,91.59,3,3.79,0.0
3,9.21,8.21,1,4,0,57,7.38,85.04,5,4.45,1.0
4,11.13,9.63,7,6,0,104,5.39,68.26,7,9.99,0.0



Unscaled Augmented Labels:


,0
0,0
1,0
2,0
3,0
4,1


In [26]:

#############################
#       SAVE AS CSV         #
#############################

save_path = "processed_data"
os.makedirs(save_path, exist_ok=True)

# Existing scaled outputs
df_X_synth.to_csv(f"{save_path}/X_synth.csv", index=False)
df_y_synth.to_csv(f"{save_path}/y_synth.csv", index=False)
X_aug.to_csv(f"{save_path}/X_train_augmented.csv", index=False)
y_aug.to_csv(f"{save_path}/y_train_augmented.csv", index=False)

# New unscaled outputs
df_X_synth_unscaled.to_csv(f"{save_path}/X_synth_unscaled.csv", index=False)
df_y_synth_unscaled.to_csv(f"{save_path}/y_synth_unscaled.csv", index=False)
X_aug_unscaled.to_csv(f"{save_path}/X_train_augmented_unscaled.csv", index=False)
y_aug_unscaled.to_csv(f"{save_path}/y_train_augmented_unscaled.csv", index=False)

print(f"Saved augmentation files to: {save_path}")


Saved augmentation files to: processed_data


In [27]:

#############################
#        OPTIONAL MAIN      #
#############################

def main():
    n_samples = 2000

    X_scaled, y_scaled = get_scaled_train_data()
    X_unscaled, y_unscaled = get_unscaled_train_data()

    if X_scaled is None or y_scaled is None:
        return

    df_X_synth_scaled, df_y_synth_scaled = generate_synthetic_data(X_scaled, y_scaled, n_samples)
    df_X_synth_unscaled_local, df_y_synth_unscaled_local = generate_unscaled_synthetic_data(
        X_unscaled, y_unscaled, n_samples
    )

    print("Scaled synthetic preview:")
    print(df_X_synth_scaled.head())
    print(df_y_synth_scaled.head())

    print("\nUnscaled synthetic preview:")
    print(df_X_synth_unscaled_local.head())
    print(df_y_synth_unscaled_local.head())

if __name__ == "__main__":
    main()


Scaled synthetic preview:
   work_hours  screen_time_hours  meetings_count  breaks_taken  \
0   -0.698164          -0.186150        0.763223     -0.973165   
1   -1.248939          -0.364158       -0.148354     -0.369748   
2    0.269760          -0.089368       -0.179965      0.065994   
3   -0.333935          -0.601934       -1.020885      1.582837   
4   -1.090191          -0.736081        0.029294     -0.335494   

   after_hours_work  app_switches  sleep_hours  task_completion  \
0         -0.293927     -1.828633     0.153969        -2.634567   
1         -0.355925     -1.714569     0.825530        -1.514109   
2         -0.863547     -0.968691     0.481465         0.367090   
3         -0.329034     -1.153279     0.712534        -0.421939   
4         -0.597638     -0.491250     1.051610         1.221635   

   isolation_index  fatigue_score  day_type_Weekend  
0        -0.720773      -1.378130         -1.490043  
1        -1.438961      -0.683498         -0.391829  
2        -0.